In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from sklearn import linear_model
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
pa_gs = pd.read_csv('data/pa_data.csv')

coastal_pa_stations = pa_gs[pa_gs['dist_atlantic_km']< 150]
lakeside_pa_stations = pa_gs[pa_gs['dist_greatlakes_km']< 150]
inland_pa_stations = pa_gs[pa_gs['dist_coast_km']>150]

In [14]:
y = ['growing_season_length']
X = ['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 'ohc700_south_atlantic', 'ohc700_south_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico','sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']
X_no_ohc = ['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico','sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']

In [46]:
def remove_nulls(data_subset, X, y):
        # Drop rows where any of the specified X columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=X, ignore_index=True)
        # Separate the cleaned dataframe back into X and y
        X_cleaned = data_subset_cleaned[X]
        y_cleaned = data_subset_cleaned[y]
        return X_cleaned, y_cleaned

def multiple_regression(data_subset, input_variables, predicted_variable):
    #performs a standardized multiple linear regression to create a model of y based on the variables within X.
    # Required Libraries: r2score, mean_absolute_error, and mean_squared_error from sklearn.metrics, 
    # train_test_split from sklearn.model_selection, linear_model from sklearn, and StandardScaler from sklearn.preprocessing

    X_cleaned, y_cleaned = remove_nulls(data_subset, input_variables, predicted_variable)

    #counts total data points included in regression
    tot_data_points = len(X_cleaned)
    
    #splits data into training and testing groups
    X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size = 0.2, random_state =0)

    #converts y_train, y_test, and y_cleaned into series
    y_train= y_train.squeeze()
    y_test= y_test.squeeze()
    y_cleaned = y_cleaned.squeeze()
    
    #scales input data to standardize to mean of 0 and standard deviation of 1
    sc = StandardScaler()
    X_train_scaled = sc.fit_transform(X_train)
    X_test_scaled = sc.fit_transform(X_test)
    
    #creates multiple regression model based on training data
    regr = linear_model.LinearRegression()
    regr.fit(X_train_scaled, y_train)

    #calculates mean absolute error, mean squared error, and r squared score based on the predicted y
    y_predicted = regr.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_predicted)
    mse = mean_squared_error(y_test, y_predicted)
    rmse = np.sqrt(mse)
    r_2_score = r2_score(y_test, y_predicted)
    n = len(y_cleaned)
    p = X_cleaned.shape[1]
    adj_r2_score = 1 - ((1 - r_2_score) * (n - 1) / (n - p - 1))
    y_std = y_cleaned.std().round(5)
    coefficients = pd.DataFrame(zip(X_cleaned.columns, X_cleaned.std().round(5), regr.coef_.round(5)))

    coefficients.columns = ['variable', 'var standard dev', 'coefficient']
    
    results = [f'MAE = {mae}', f'MSE = {mse}', f'RMSE = {rmse}', f'R Squared = {r_2_score}', 
               f'Adj. R Squared = {adj_r2_score}', f'Total Data Points = {tot_data_points}',
               f'(Reference) predicted variable standard deviation = {y_std}', coefficients]
    
    return display(results)

In [47]:
multiple_regression(pa_gs, X, y)

['MAE = 8.653954401679119',
 'MSE = 118.54336216146942',
 'RMSE = 10.887762036409017',
 'R Squared = 0.802483321255483',
 'Adj. R Squared = 0.7956683495914842',
 'Total Data Points = 1740',
 '(Reference) predicted variable standard deviation = 25.12848',
                       variable  var standard dev  coefficient
 0                   dtr_annual           1.16262     -9.44840
 1                   dtr_spring           1.46341     -0.78335
 2                   dtr_summer           1.35145      1.65706
 3                  tmax_annual           1.65028      8.41794
 4               prcp_annual_mm         224.68173    -16.30678
 5       prcp_growing_season_mm         184.12341     20.47930
 6               prcp_spring_mm          85.87998      3.16700
 7                     latitude           0.67241    -11.85397
 8                    longitude           1.74971      7.59368
 9                  elevation_m         158.32652     -1.45898
 10               dist_coast_km          60.52147   

In [33]:
multiple_regression(pa_gs, X_no_ohc, y)

['MAE = 10.25825018564257',
 'MSE = 174.2708727493307',
 'RMSE = 13.201169370526639',
 'R Squared = 0.8111050764873684',
 'Adj. R Squared = 0.8098694381568057',
 'Total Data Points = 7233',
 '(Reference) predicted variable standard deviation = 31.23174',
                       variable  var standard dev  coefficient
 0                   dtr_annual           1.42052    -13.66920
 1                   dtr_spring           1.62191      0.44842
 2                   dtr_summer           1.85314      1.49569
 3                  tmax_annual           1.73240      7.83147
 4               prcp_annual_mm         210.58578    -14.00453
 5       prcp_growing_season_mm         170.51672     19.66358
 6               prcp_spring_mm          76.52566      3.04974
 7                     latitude           0.65895     -8.61432
 8                    longitude           1.76333      8.83282
 9                  elevation_m         164.23514     -4.29876
 10               dist_coast_km          60.51768   

In [34]:
y_lsf = 'last_spring_frost_doy'
multiple_regression(pa_gs, X, y_lsf)

['MAE = 6.614278668670575',
 'MSE = 71.88736803306439',
 'RMSE = 8.478641874325415',
 'R Squared = 0.7215828868957804',
 'Adj. R Squared = 0.711976585551316',
 'Total Data Points = 1740',
 '(Reference) predicted variable standard deviation = 16.21068',
                       variable  var standard dev  coefficient
 0                   dtr_annual           1.16262      1.24219
 1                   dtr_spring           1.46341      4.57903
 2                   dtr_summer           1.35145     -0.60851
 3                  tmax_annual           1.65028     -4.65707
 4               prcp_annual_mm         224.68173      9.58390
 5       prcp_growing_season_mm         184.12341    -11.38340
 6               prcp_spring_mm          85.87998     -1.90406
 7                     latitude           0.67241      6.09500
 8                    longitude           1.74971     -4.31773
 9                  elevation_m         158.32652     -0.12730
 10               dist_coast_km          60.52147     

In [35]:
multiple_regression(pa_gs, X_no_ohc, y_lsf)

['MAE = 7.6292215766300435',
 'MSE = 90.8392731817753',
 'RMSE = 9.530963916717726',
 'R Squared = 0.7080585032477958',
 'Adj. R Squared = 0.7061487954750256',
 'Total Data Points = 7233',
 '(Reference) predicted variable standard deviation = 17.91329',
                       variable  var standard dev  coefficient
 0                   dtr_annual           1.42052      5.13831
 1                   dtr_spring           1.62191      1.41196
 2                   dtr_summer           1.85314     -0.78730
 3                  tmax_annual           1.73240     -3.97339
 4               prcp_annual_mm         210.58578      7.21697
 5       prcp_growing_season_mm         170.51672     -9.99612
 6               prcp_spring_mm          76.52566     -1.30833
 7                     latitude           0.65895      3.18691
 8                    longitude           1.76333     -1.86645
 9                  elevation_m         164.23514      1.54282
 10               dist_coast_km          60.51768    

In [36]:
y_fff = 'first_fall_frost_doy'
multiple_regression(pa_gs, X, y_fff)

['MAE = 5.7644925364908515',
 'MSE = 60.33006450796742',
 'RMSE = 7.767243044218935',
 'R Squared = 0.5923504186978129',
 'Adj. R Squared = 0.5782851743697184',
 'Total Data Points = 1740',
 '(Reference) predicted variable standard deviation = 12.9393',
                       variable  var standard dev  coefficient
 0                   dtr_annual           1.16262     -8.20621
 1                   dtr_spring           1.46341      3.79568
 2                   dtr_summer           1.35145      1.04855
 3                  tmax_annual           1.65028      3.76087
 4               prcp_annual_mm         224.68173     -6.72288
 5       prcp_growing_season_mm         184.12341      9.09590
 6               prcp_spring_mm          85.87998      1.26293
 7                     latitude           0.67241     -5.75897
 8                    longitude           1.74971      3.27595
 9                  elevation_m         158.32652     -1.58627
 10               dist_coast_km          60.52147    

In [37]:
multiple_regression(pa_gs, X_no_ohc, y_fff)

['MAE = 8.284513830061027',
 'MSE = 116.53790954058067',
 'RMSE = 10.79527255517806',
 'R Squared = 0.6055048570124397',
 'Adj. R Squared = 0.6029243042329804',
 'Total Data Points = 7233',
 '(Reference) predicted variable standard deviation = 17.70698',
                       variable  var standard dev  coefficient
 0                   dtr_annual           1.42052     -8.53089
 1                   dtr_spring           1.62191      1.86038
 2                   dtr_summer           1.85314      0.70839
 3                  tmax_annual           1.73240      3.85808
 4               prcp_annual_mm         210.58578     -6.78756
 5       prcp_growing_season_mm         170.51672      9.66746
 6               prcp_spring_mm          76.52566      1.74140
 7                     latitude           0.65895     -5.42741
 8                    longitude           1.76333      6.96637
 9                  elevation_m         164.23514     -2.75594
 10               dist_coast_km          60.51768   

In [39]:
psu_gs = pa_gs[pa_gs['station_name']=='STATE COLLEGE']
X_psu_no_ohc = ['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico','sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']
multiple_regression(psu_gs, X_psu_no_ohc, y)

['MAE = 19.468445087767137',
 'MSE = 636.5007399873906',
 'RMSE = 25.22896628852222',
 'R Squared = -0.7318386641976793',
 'Adj. R Squared = -3.0986848386011747',
 'Total Data Points = 72',
 '(Reference) predicted variable standard deviation = 17.83043',
                       variable  var standard dev  coefficient
 0                   dtr_annual           0.73846    -21.19920
 1                   dtr_spring           0.92561      8.52579
 2                   dtr_summer           1.13625     20.64721
 3                  tmax_annual           0.75939      3.05474
 4               prcp_annual_mm         179.45444    -17.85343
 5       prcp_growing_season_mm         148.24445     32.27103
 6               prcp_spring_mm          63.41483      3.16547
 7                   oni_annual           0.61088     -4.26255
 8                   nao_annual           0.37804     -0.28690
 9                      nao_djf           0.83750      4.51966
 10                  pna_annual           0.35267   